<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/02d_processing_nbhd_scatterplot_demographics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **This notebook includes code for the scatterplot of business resiliency by neighborhood and pulls in demographic data by neighborhood.**

## Set Up

In [ ]:
!pip install contextily

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

#added more that we use in lab
import os
%matplotlib inline
import matplotlib.pyplot as plt
from shapely.geometry import LineString


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive
!ls /content/drive/Shareddrives

In [ ]:
import pandas as pd

gdf = gpd.read_file("/content/drive/MyDrive/Courses/Spring 2026/Urban Informatics/C255_final_project/cleaned/open_close_after_2016.geojson")

In [ ]:
gdf.info()

In [ ]:
# # Importing SF geometry
# # URL for 2025 TIGER/Line Place boundaries - info here on
# ## how to use: https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html
url = "https://www2.census.gov/geo/tiger/TIGER2025/PLACE/tl_2025_06_place.zip"

places = gpd.read_file(url)

# Filtered to SF
sf_poly = places[
    (places["NAME"] == "San Francisco") &
    (places["STATEFP"] == "06")   # 06 = California
]


# project to same as our gdf
sf_poly = sf_poly.to_crs(epsg=4326)

## Plotting Closings vs. Openings over time for all businesses

In [ ]:
import plotly.express as px

# Counting number of opening and closings
counts = gdf.groupby(["year", "status"]).size().reset_index(name="count")
counts = counts[counts["year"] <= 2025]

# Plotting
fig = px.line(
    counts,
    x="year",
    y="count",
    color="status",
    markers=True
)

# Adding labels
fig.update_layout(
    title="San Francisco Business Openings vs Closings (2016–2025)",
    xaxis_title="Year",
    yaxis_title="Number of Businesses",
    legend_title="Status"
)

fig.update_xaxes(dtick=1)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="lightgray"
)

fig.show()


## Neighborhood Level Analysis

In [ ]:
# Adding SF neighborhood geometry

nbhd_gdf = gpd.read_file(
    "/content/drive/MyDrive/Courses/Spring 2026/Urban Informatics/C255_final_project/raw/sf_neighborhoods.geojson"
)

In [ ]:
nbhd_gdf.columns

In [ ]:
# Joining the gdf and nbhd gdf

gdf = gdf.set_crs(epsg=4326)
nbhd_gdf = nbhd_gdf.to_crs(epsg=4326)

# joining within
business_neighborhood_gdf = gpd.sjoin(gdf, nbhd_gdf, how="left", predicate="within")

In [ ]:
business_neighborhood_gdf.columns

In [ ]:
gdf = business_neighborhood_gdf.copy()

# filtering for 2019 to 2025
gdf = gdf[gdf['year'].between(2019, 2025)]

# openings
openings = (
    gdf[gdf['status'] == 'opened']
    .groupby(['nhood', 'year'])
    .size()
    .unstack(fill_value=0)
)

# closings
closings = (
    gdf[gdf['status'] == 'closed']
    .groupby(['nhood', 'year'])
    .size()
    .unstack(fill_value=0)
)

all_years = range(2019, 2026)
openings = openings.reindex(columns=all_years, fill_value=0)
closings = closings.reindex(columns=all_years, fill_value=0)

# renaming columns to be open_year
openings.columns = [f"open_{y}" for y in openings.columns]
closings.columns = [f"close_{y}" for y in closings.columns]

# combining openings and closings to get a table to compare across neighborhoods
neighborhood_table = openings.join(closings, how='outer').fillna(0).astype(int)

neighborhood_table

# Baseline by number of businesses in each neighborhood

In [ ]:
gdf = business_neighborhood_gdf.copy()

# baseline by total # of businesses
baseline = gdf.groupby('nhood')['uniqueid'].nunique()

# closures during COVID (2020–2021)
closures = gdf[
    (gdf['year'].between(2020, 2021)) &
    (gdf['status'] == 'closed')
].groupby('nhood').size()

# openings during recovery (2022–2025)
openings = gdf[
    (gdf['year'].between(2022, 2025)) &
    (gdf['status'] == 'opened')
].groupby('nhood').size()

# combine
rate_table = pd.DataFrame({
    'baseline': baseline,
    'closures_2020_21': closures,
    'openings_2022_25': openings
}).fillna(0)

# adding rates
rate_table['closure_rate'] = rate_table['closures_2020_21'] / rate_table['baseline']
rate_table['recovery_rate'] = rate_table['openings_2022_25'] / rate_table['baseline']

rate_table

## Calculate Business Resilience Rate

In [ ]:
# Filtering to nbhds with at least 50 businesses
rate_table_filtered = rate_table[rate_table['baseline'] >= 50]

In [ ]:
df = rate_table_filtered.copy()
df['resilience'] = df['recovery_rate'] - df['closure_rate']

top_10 = df.sort_values('resilience', ascending=False).head(10)
bottom_10 = df.sort_values('resilience', ascending=True).head(10)

print("Most Resilient Neighborhoods")
display(top_10)

print("Least Resilient Neighborhoods")
display(bottom_10)

## Create scatter plot to select by neighborhood

In [ ]:
import plotly.graph_objects as go

df = rate_table_filtered.copy()

neighborhoods = df.index.tolist()

base_color = "lightgray"
highlight_color = "red"

x_mean = df["closure_rate"].mean()
y_mean = df["recovery_rate"].mean()

x_min, x_max = df["closure_rate"].min(), df["closure_rate"].max()
y_min, y_max = df["recovery_rate"].min(), df["recovery_rate"].max()

x_pad = (x_max - x_min) * 0.08
y_pad = (y_max - y_min) * 0.08

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df["closure_rate"],
        y=df["recovery_rate"],
        mode="markers",
        text=df.index,
        customdata=df[["closures_2020_21", "openings_2022_25"]].values,
        hovertemplate=
            "<b>%{text}</b><br>" +
            "Closure rate: %{x:.3f}<br>" +
            "Recovery rate: %{y:.3f}<br>" +
            "Total closures: %{customdata[0]:.0f}<br>" +
            "Total openings: %{customdata[1]:.0f}<br>" +
            "<extra></extra>",
        marker=dict(color=base_color, size=10),
    )
)

# Quadrant lines
fig.add_shape(type="line", x0=x_mean, x1=x_mean, y0=y_min, y1=y_max,
              line=dict(dash="dash", color="black"))
fig.add_shape(type="line", x0=x_min, x1=x_max, y0=y_mean, y1=y_mean,
              line=dict(dash="dash", color="black"))

# Quadrant labels
quadrant_labels = [
    dict(x=x_max - x_pad, y=y_max - y_pad, text="High closure<br>High recovery"),
    dict(x=x_min + x_pad, y=y_max - y_pad, text="Low closure<br>High recovery"),
    dict(x=x_min + x_pad, y=y_min + y_pad, text="Low closure<br>Low recovery"),
    dict(x=x_max - x_pad, y=y_min + y_pad, text="High closure<br>Low recovery"),
]
for q in quadrant_labels:
    fig.add_annotation(
        x=q["x"], y=q["y"],
        text=q["text"],
        showarrow=False,
        font=dict(size=10, color="gray"),
        align="center",
        bgcolor="rgba(255,255,255,0.6)",
        borderpad=3
    )

# Dropdown
buttons = [
    dict(
        label="All",
        method="update",
        args=[
            {"marker": {"color": [base_color] * len(neighborhoods), "size": 10}},
            {"title": "COVID Impact vs Recovery"}
        ],
    )
]

for n in neighborhoods:
    colors = [highlight_color if name == n else base_color for name in neighborhoods]
    sizes  = [14 if name == n else 10 for name in neighborhoods]
    buttons.append(
        dict(
            label=n,
            method="update",
            args=[
                {"marker": {"color": colors, "size": sizes}},
                {"title": f"Highlight: {n}"}
            ],
        )
    )

fig.update_layout(
    title="COVID Impact vs Recovery",
    xaxis_title="Closure Rate (2020–2021)",
    yaxis_title="Recovery Rate (2022–2025)",
    updatemenus=[dict(buttons=buttons, direction="down", showactive=True)]
)

fig.show()


# Demographic Data by Neighborhood

In [ ]:
import geopandas as gpd
from shapely import wkt

# Read in tract to neighborhood geometry file
neighborhoods = pd.read_csv('/content/drive/MyDrive/Courses/Spring 2026/Urban Informatics/C255_final_project/raw/tract_to_neighborhood_geom.csv')

# Convert WKT geometry column to shapely geometry and make GeoDataFrame
neighborhoods["geometry"] = neighborhoods["the_geom"].apply(wkt.loads)
neighborhoods = gpd.GeoDataFrame(neighborhoods, geometry="geometry")

In [ ]:
neighborhoods = neighborhoods[[
    "geoid",
    "neighborhoods_analysis_boundaries",
    "geometry"
]].rename(columns={
    "geoid": "tract",
    "neighborhoods_analysis_boundaries": "neighborhood"
})

In [ ]:
neighborhoods.head()

In [ ]:
!pip install census
from census import Census
import pandas as pd

# Prepare Census Bureau API for data pulling
api_key = 'c9ad2fda3c683e4a14992e89daed0afee55738d2'
c = Census(key=api_key)

In [ ]:
race_vars = {
    "B03002_001E": "population",
    "B03002_003E": "white",           # Non-Hispanic White
    "B03002_004E": "black",           # Non-Hispanic Black
    "B03002_005E": "aian",            # Non-Hispanic American Indian/Alaska Native
    "B03002_006E": "asian",           # Non-Hispanic Asian
    "B03002_007E": "nhpi",            # Non-Hispanic Native Hawaiian/Pacific Islander
    "B03002_008E": "other",           # Non-Hispanic Some Other Race
    "B03002_012E": "latina_o",        # Hispanic or Latino
}

income_vars = {
    "B19013_001E": "median_income"
}

all_vars = {**race_vars, **income_vars}

# Pulling ACS 5-year 2023 data
acs = pd.DataFrame(
    c.acs5.get(
        list(all_vars.keys()),
        {'for': 'tract:*', 'in': 'state:06 county:075'},
        year=2023
    )
).rename(columns=all_vars)

acs['tract'] = '06075' + acs['tract']

In [ ]:
# Replace Census null codes with NaN
acs['median_income'] = acs['median_income'].replace(-666666666, pd.NA)

In [ ]:
acs.head()

In [ ]:
neighborhoods.head()

In [ ]:
acs['tract'] = acs['tract'].astype(str).str.lstrip('0')

neighborhoods['tract'] = neighborhoods['tract'].astype(str)
acs['tract'] = acs['tract'].astype(str)

merged = neighborhoods.merge(acs, on='tract', how='inner')

# Merge with neighborhoods shapefile
merged = neighborhoods.merge(acs, on='tract', how='inner')

# Aggregate by neighborhood
agg_cols = {
    'population': 'sum',
    'white': 'sum',
    'black': 'sum',
    'aian': 'sum',
    'asian': 'sum',
    'nhpi': 'sum',
    'other': 'sum',
    'latina_o': 'sum',
    'median_income': 'mean'
}

final = merged.groupby('neighborhood').agg(agg_cols).reset_index()



In [ ]:
# Percentages
# combining Native American into other because all the values are less than 1%
# combining NHPI with Asian because all the values are less than .02%
final['other'] = final['other'] + final['aian']
final = final.drop(columns='aian')
final['asian'] = final['asian'] + final['nhpi']
final = final.drop(columns='nhpi')

final['pct_other']   = final['other']   / final['population']
final['pct_white']   = final['white']   / final['population']
final['pct_black']   = final['black']   / final['population']
final['pct_asian']   = final['asian']   / final['population']
final['pct_latina_o']= final['latina_o']/ final['population']

# Filter out very small nbhds
final_df = final[final['population'] >= 200]

final_df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, Output, VBox

out = Output()

def plot_neighborhood(neighborhood):
    row = final[final["neighborhood"] == neighborhood].iloc[0]

    race_cols   = ["pct_white", "pct_black", "pct_asian", "pct_latina_o", "pct_other"]
    race_labels = ["White (NH)", "Black (NH)", "Asian & Pacific Islander (NH)", "Hispanic/Latino", "Other (NH)"]
    values = [row[c] * 100 for c in race_cols]
    income = row["median_income"]
    income_str = f"${income:,.0f}" if pd.notna(income) else "No data"

    with out:
        out.clear_output(wait=True)

        fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [3, 1]})
        fig.suptitle("Race/Ethnicity by Neighborhood", fontsize=13, fontweight="bold")
        ax.set_title(neighborhood, fontsize=11, color="gray", pad=8)

        bars = ax.bar(race_labels, values, color="#378ADD", width=0.5)
        ax.set_ylabel("% of population")
        ax.set_ylim(0, max(values) + 10)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                    f"{val:.1f}%", ha="center", va="bottom", fontsize=9)
        ax.tick_params(axis='x', labelsize=9)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        # Right panel — income as text
        ax2.axis("off")
        ax2.text(0.5, 0.6, "Median\nhousehold\nincome", ha="center", va="center",
                 fontsize=11, color="gray", transform=ax2.transAxes, linespacing=1.6)
        ax2.text(0.5, 0.35, income_str, ha="center", va="center",
                 fontsize=18, fontweight="bold", color="#378ADD", transform=ax2.transAxes)

        plt.tight_layout()
        plt.show()

dropdown = Dropdown(options=sorted(final["neighborhood"].tolist()), description="Neighborhood:")
dropdown.observe(lambda change: plot_neighborhood(change["new"]), names="value")

plot_neighborhood(dropdown.value)
display(VBox([dropdown, out]))

In [ ]:
# Set up to plot resilience rate by demographic variables

df = rate_table_filtered.copy()
df['resilience'] = df['recovery_rate'] - df['closure_rate']

# Merge in demographics
final_indexed = final.set_index('neighborhood')
final_indexed['pct_poc'] = 1 - final_indexed['pct_white']
demo_cols = ['median_income', 'pct_poc']
df = df.join(final_indexed[demo_cols], how='left')
df['median_income'] = pd.to_numeric(df['median_income'], errors='coerce')

df = df.sort_values('resilience', ascending=False)


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Resilience vs Median Income", "Resilience vs % People of Color"),
    horizontal_spacing=0.12
)

hover_income = (
    "<b>%{text}</b><br>"
    "Resilience: %{y:.3f}<br>"
    "Median Income: $%{x:,.0f}<br>"
    "<extra></extra>"
)

hover_poc = (
    "<b>%{text}</b><br>"
    "Resilience: %{y:.3f}<br>"
    "% POC: %{x:.1%}<br>"
    "<extra></extra>"
)

# Chart 1: Resilience vs Median Income
fig.add_trace(go.Scatter(
    x=df['median_income'],
    y=df['resilience'],
    mode='markers+text',
    text=df.index,
    textposition='top center',
    textfont=dict(size=7, color='gray'),
    hovertemplate=hover_income,
    marker=dict(
        color=df['median_income'],
        colorscale='RdBu',
        size=10,
        showscale=True,
        colorbar=dict(title='Median Income', x=0.44),
        line=dict(width=1, color='white')
    ),
    showlegend=False
), row=1, col=1)

# Chart 2: Resilience vs % POC
fig.add_trace(go.Scatter(
    x=df['pct_poc'],
    y=df['resilience'],
    mode='markers+text',
    text=df.index,
    textposition='top center',
    textfont=dict(size=7, color='gray'),
    hovertemplate=hover_poc,
    marker=dict(
        color=df['pct_poc'],
        colorscale='RdBu_r',
        size=10,
        showscale=True,
        colorbar=dict(title='% POC', x=1.02),
        line=dict(width=1, color='white')
    ),
    showlegend=False
), row=1, col=2)

fig.update_xaxes(title_text="Median Income", tickprefix="$", tickformat=",", col=1)
fig.update_xaxes(title_text="% People of Color", tickformat=".0%", col=2)
fig.update_yaxes(title_text="Resilience (recovery rate − closure rate)", col=1)
fig.update_yaxes(title_text="", col=2)

fig.update_layout(
    title="Business Resilience by Neighborhood Demographics",
    width=1200,
    height=560,
    plot_bgcolor="white",
)

fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# --- Build net change per year per neighborhood ---
years = list(range(2019, 2026))

net_change = pd.DataFrame(index=neighborhood_table.index)
for y in years:
    open_col  = f"open_{y}"
    close_col = f"close_{y}"
    if open_col in neighborhood_table.columns and close_col in neighborhood_table.columns:
        net_change[y] = neighborhood_table[open_col] - neighborhood_table[close_col]

# --- Merge in demographics ---
final_indexed = final.set_index('neighborhood')
final_indexed['pct_poc'] = 1 - final_indexed['pct_white']

net_change = net_change.join(final_indexed[['median_income', 'pct_poc']], how='inner')
net_change['median_income'] = pd.to_numeric(net_change['median_income'], errors='coerce')

# --- Assign quartiles ---
net_change['income_quartile'] = pd.qcut(
    net_change['median_income'], q=4,
    labels=['Q1 (lowest income)', 'Q2', 'Q3', 'Q4 (highest income)']
)
net_change['poc_quartile'] = pd.qcut(
    net_change['pct_poc'], q=4,
    labels=['Q1 (least POC)', 'Q2', 'Q3', 'Q4 (most POC)']
)

# --- Average net change per year by quartile ---
income_trends = (
    net_change.groupby('income_quartile', observed=True)[years]
    .mean()
)
poc_trends = (
    net_change.groupby('poc_quartile', observed=True)[years]
    .mean()
)

# --- Colors ---
income_palette = ['#d6604d', '#f4a582', '#92c5de', '#2166ac']
poc_palette    = ['#2166ac', '#92c5de', '#f4a582', '#d6604d']

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Avg Net Business Change by Income Quartile",
        "Avg Net Business Change by % POC Quartile"
    ),
    horizontal_spacing=0.12
)

# --- Chart 1: Income quartiles ---
for i, (q_label, row) in enumerate(income_trends.iterrows()):
    fig.add_trace(go.Scatter(
        x=years,
        y=row.values,
        mode='lines+markers',
        name=str(q_label),
        legendgroup=f"income_{q_label}",
        line=dict(color=income_palette[i], width=2.5),
        marker=dict(size=7),
        hovertemplate=f"<b>{q_label}</b><br>Year: %{{x}}<br>Avg net change: %{{y:.1f}}<extra></extra>"
    ), row=1, col=1)

# --- Chart 2: POC quartiles ---
for i, (q_label, row) in enumerate(poc_trends.iterrows()):
    fig.add_trace(go.Scatter(
        x=years,
        y=row.values,
        mode='lines+markers',
        name=str(q_label),
        legendgroup=f"poc_{q_label}",
        line=dict(color=poc_palette[i], width=2.5),
        marker=dict(size=7),
        showlegend=True,
        hovertemplate=f"<b>{q_label}</b><br>Year: %{{x}}<br>Avg net change: %{{y:.1f}}<extra></extra>"
    ), row=1, col=2)

# COVID reference line on both charts
for col in [1, 2]:
    fig.add_shape(
        type='line', x0=2020, x1=2020,
        y0=-50, y1=50,
        line=dict(dash='dash', color='black', width=1),
        row=1, col=col
    )
    fig.add_annotation(
        x=2020, y=45,
        text="COVID", showarrow=False,
        font=dict(size=9, color='black'),
        xref=f"x{'' if col==1 else col}",
        yref=f"y{'' if col==1 else col}"
    )
    # Zero line
    fig.add_shape(
        type='line', x0=2019, x1=2025,
        y0=0, y1=0,
        line=dict(dash='dot', color='gray', width=1),
        row=1, col=col
    )

fig.update_xaxes(tickvals=years, ticktext=[str(y) for y in years])
fig.update_yaxes(title_text="Avg net change (openings − closings)", col=1)

fig.update_layout(
    title="Pre vs Post COVID Business Trajectory by Demographic Quartile",
    width=1200,
    height=520,
    plot_bgcolor="white",
    legend=dict(orientation="v", x=1.02, y=0.5)
)

fig.show()